# Assignment 01: ResNet 18 vs ViT

**Available:** Aug 19, 2025 4:30pm until Aug 28, 2025 11:59pm

## Tasks:

1. Train on Human Action Recognition (40%) https://www.kaggle.com/datasets/shashankrapolu/human-Links to an external site.action-recognition-dataset/dataLinks to an external site.​
2. Evaluation on test set human-action-recognition​: 15 classes, 218 MB, 12600 images, train/test split available​
3. ResNet 18 vs ViT​
4. Report and code zip (10%)​
5. Errors and obstacles faced running the model​
6. Must have conda requirement.txt, cli commands to generate the evaluation results above (random checks will be perform to verify)​
7. Training log​
8. Video of live demo (20%)​
9. The video should comprise of visual outputs (e.g., a text "cat" overlay on the image being classified that has a cat in it)​
10. Insights (30%)​
11. ResNet vs ViT​ Computational comparisons​
12. Why one is better ...


### Import Libraries

In [ ]:
## Import Libraries

# Set CUDA_VISIBLE_DEVICES to make both GPUs visible
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'

import torch
import torch.nn as nn
import torchvision
import cv2
import matplotlib.pyplot as plt
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch import optim
from tqdm.notebook import tqdm
from torchinfo import summary
import einops
import PIL
import numpy as np
import pandas as pd

# Install einops for tensor manipulation
%pip install einops

# Comprehensive GPU diagnostics
print("=== GPU Diagnostics ===")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"Number of GPUs detected: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print("\n=== All Available GPUs ===")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}:")
        print(f"  Name: {props.name}")
        print(f"  Total Memory: {props.total_memory / 1024**3:.2f} GB")
        print(f"  Multi-processor count: {props.multi_processor_count}")
        print(f"  Compute Capability: {props.major}.{props.minor}")
        print()

# Device selection with preference for cuda:1 (A6000) -> cuda:0 (4090) -> cpu
if torch.cuda.is_available() and torch.cuda.device_count() > 1:
    device = torch.device('cuda:1')  # This should now be your A6000!
    print(f"Using GPU 1: {torch.cuda.get_device_name(1)}")
elif torch.cuda.is_available():
    device = torch.device('cuda:0')
    print(f"Using GPU 0: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device('cpu')
    print("Using CPU")

print(f"Selected device: {device}")

/home/malneyugnfl/anaconda3/envs/huggingface/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/home/malneyugnfl/anaconda3/envs/huggingface/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


Note: you may need to restart the kernel to use updated packages.
=== GPU Diagnostics ===
PyTorch version: 2.6.0
CUDA available: False
CUDA version: None
Number of GPUs detected: 0
Using CPU
Selected device: cpu


### Import and Process Human Action Recognition Dataset

In [6]:
## Import dataset and process it

class ActionRecognitionDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        """
        Custom dataset for Human Action Recognition with labels
        Args:
            csv_file (string): Path to the csv file with annotations.
            root_dir (string): Directory with all the images.
            transform (callable, optional): Optional transform to be applied on a sample.
        """
        import pandas as pd
        
        self.annotations = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform
        
        # Get unique class names and create class-to-index mapping
        self.classes = sorted(self.annotations['label'].unique())
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}
        
        print(f"Found {len(self.classes)} classes: {self.classes}")
        
    def __len__(self):
        return len(self.annotations)
    
    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()
            
        # Get image filename and label from CSV
        img_name = self.annotations.iloc[idx, 0]  # filename column
        label_name = self.annotations.iloc[idx, 1]  # label column
        
        # Convert label name to index
        label = self.class_to_idx[label_name]
        
        # Construct full image path
        img_path = os.path.join(self.root_dir, img_name)
        
        # Load image
        image = PIL.Image.open(img_path)
        
        # Convert RGBA to RGB if necessary
        if image.mode == 'RGBA':
            image = image.convert('RGB')
        
        # Apply transforms
        if self.transform:
            image = self.transform(image)
        
        return image, label

class TestDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        """
        Test dataset for Human Action Recognition without labels (for prediction)
        Args:
            csv_file (string): Path to the csv file with filenames only.
            root_dir (string): Directory with all the images.
            transform (callable, optional): Optional transform to be applied on a sample.
        """
        import pandas as pd
        
        self.annotations = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform
        
        print(f"Found {len(self.annotations)} test images")
        
    def __len__(self):
        return len(self.annotations)
    
    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()
            
        # Get image filename from CSV
        img_name = self.annotations.iloc[idx, 0]  # filename column
        
        # Construct full image path
        img_path = os.path.join(self.root_dir, img_name)
        
        # Load image
        image = PIL.Image.open(img_path)
        
        # Convert RGBA to RGB if necessary
        if image.mode == 'RGBA':
            image = image.convert('RGB')
        
        # Apply transforms
        if self.transform:
            image = self.transform(image)
        
        return image, img_name  # Return image and filename for prediction

# Data transforms with proper normalization for RGB images
data_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),  # Converts PIL to tensor and scales to [0,1]
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet normalization
])

# Define data augmentation for training
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])



In [7]:
# Create datasets
train_data = ActionRecognitionDataset(
    csv_file='data/Training_set.csv',
    root_dir='data/train',
    transform=train_transform
)

# For this assignment, we'll create validation data from training data
# Split training data into train and validation (80-20 split)
from torch.utils.data import random_split

train_size = int(0.8 * len(train_data))
val_size = len(train_data) - train_size
train_subset, val_subset = random_split(train_data, [train_size, val_size])

# Create validation dataset with test transforms (no augmentation)
val_data = ActionRecognitionDataset(
    csv_file='data/Training_set.csv',
    root_dir='data/train',
    transform=data_transform
)

# Extract validation indices for proper subset
val_dataset = torch.utils.data.Subset(val_data, val_subset.indices)

# Create test dataset and loader for final evaluation
test_data = TestDataset(
    csv_file='data/Testing_set.csv',
    root_dir='data/test',
    transform=data_transform  # No augmentation for test data
)

print(f"\nDataset Information:")
print(f"Total training samples: {len(train_data)}")
print(f"Training subset: {len(train_subset)}")
print(f"Validation subset: {len(val_dataset)}")
print(f"Number of classes: {len(train_data.classes)}")
print(f"Classes: {train_data.classes}")
print(f"\nTest Dataset Information:")
print(f"Test samples: {len(test_data)}")


# Test loading a sample
try:
    sample_image, sample_label = train_data[0]
    print(f"\nSample check:")
    print(f"Image shape: {sample_image.shape}")
    print(f"Label: {sample_label} ({train_data.classes[sample_label]})")
except Exception as e:
    print(f"Error loading sample: {e}")
    print("Please check if the image files exist in the correct directory")

# Test loading from test dataset
try:
    test_sample_image, test_filename = test_data[0]
    print(f"\nTest Sample Check:")
    print(f"Test image shape: {test_sample_image.shape}")
    print(f"Test filename: {test_filename}")
    print("Test dataset loading successful!")
except Exception as e:
    print(f"Error loading test sample: {e}")
    print("Please check if the test image files exist in the correct directory")

### Create Dataloaders that will be used for ResNet 18 and ViT

In [ ]:
# Create data loaders
batch_size = 50
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=4)


print(f"\nData Loaders:")
print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

### Import ResNet 18

In [ ]:
resnet18= torchvision.models.resnet18(weights='IMAGENET1K_V1')
num_ftrs = resnet18.fc.in_features
resnet18.fc = nn.Linear(num_ftrs, len(train_data.classes))  # Adjust final layer for 15 classes
resnet18 = resnet18.to(device)

In [ ]:
# Information for ResNet-18

print('Res18 trainable parameters in millions:',sum(p.numel() for p in resnet18.parameters() if p.requires_grad)/1000000)
print('ResNet-18 Model Summary:')

# The input size includes: (batchsize, channels, height, width)
print(summary(resnet18, input_size=(1, 3, 224, 224)))

In [ ]:
!nvidia-smi

### Import ViT

In [ ]:
# Import and Setup ViT

# Import and Setup ViT
# Install timm (PyTorch Image Models) for Vision Transformer
%pip install timm

import timm
import torch
import torch.nn as nn

# Define number of classes for Human Action Recognition
num_classes = 

print("=== Setting up Vision Transformer ===")

# Load a pre-trained Vision Transformer
# Using ViT-Base/16 which is a good balance of performance and computational efficiency
vit_model = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=num_classes)

# Move model to device (CPU for now since CUDA is not available)
device = torch.device('cpu')
vit_model = vit_model.to(device)

print(f"✅ ViT Model loaded successfully!")
print(f"Model: {vit_model.__class__.__name__}")
print(f"Number of classes: {num_classes}")
print(f"Device: {device}")

# Get model information
vit_params = sum(p.numel() for p in vit_model.parameters() if p.requires_grad)
print(f'ViT trainable parameters in millions: {vit_params/1000000:.2f}')

# Create a sample input to test the model
print(f"\n=== Testing ViT with Sample Data ===")
try:
    # Create a dummy batch (batch_size=2, channels=3, height=224, width=224)
    sample_input = torch.randn(2, 3, 224, 224).to(device)
    print(f"Sample input shape: {sample_input.shape}")
    
    # Test forward pass
    with torch.no_grad():
        vit_output = vit_model(sample_input)
    
    print(f"ViT output shape: {vit_output.shape}")
    print(f"Expected shape: (batch_size=2, num_classes={num_classes})")
    print("✅ ViT is ready for training!")
    
    # Display model architecture summary
    print(f"\n=== ViT Architecture Summary ===")
    print(f"Model type: Vision Transformer Base (patch size 16)")
    print(f"Input size: 224x224 pixels") 
    print(f"Patch size: 16x16 pixels")
    print(f"Number of patches: {(224//16)**2} patches")
    print(f"Embedding dimension: 768")
    print(f"Number of attention heads: 12")
    print(f"Number of transformer layers: 12")
    print(f"Output classes: {num_classes}")
    
except Exception as e:
    print(f"❌ Error testing ViT: {e}")

print(f"\n🎯 ViT is now ready to be compared with ResNet-18!")
print(f"Both models will classify images into {num_classes} human action categories:")

### Run Training for ViT and ResNet-18

In [ ]:
# Run Training for Vit and ResNet-18

In [ ]:
# Compare results and do analysis